In [2]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.llms.langchain import LangChainLLM
from langchain_openai.chat_models import ChatOpenAI
from Config.load_key import open_key
from Config.load_key import base_model
from Config.load_key import base_url

model = LangChainLLM(
    ChatOpenAI(
        model=base_model,
        base_url=base_url,
        api_key=open_key
    )
)
agent = ReActAgent(llm=model)
ctx = Context(agent)
resp = await agent.run("怎么退款?", ctx=ctx)
resp


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='退款的具体流程取决于您购买商品或服务的平台或商家。通常，您可以按照以下一般步骤操作：\n\n1. **登录账户**：进入您购买商品或服务的平台（如淘宝、京东、App Store 等）并登录您的账户。\n2. **找到订单**：在“我的订单”或类似页面中找到需要退款的订单。\n3. **申请退款**：点击“申请退款”或“退货/退款”，根据提示选择退款原因并提交申请。\n4. **等待审核**：商家或平台会审核您的退款请求，可能需要您提供相关凭证（如照片、聊天记录等）。\n5. **处理完成**：审核通过后，退款会按原支付路径退回，具体到账时间视支付方式而定。\n\n如果您能提供更具体的场景（比如是哪个平台或服务），我可以给出更有针对性的建议！')]), structured_response=None, current_agent_name='Agent', raw=None, tool_calls=[], retry_messages=[])

Memory和tools是大模型语言原生支持的功能

In [ ]:
from llama_index.core.memory.chat_memory_buffer import ChatMemoryBuffer
from llama_index.core.tools import FunctionTool, query_engine


def get_weather(city: str) -> str:
    """获取某个城市的天气"""
    return f"城市:{city},天气一直都是晴天"


# weather_tools = FunctionTool.from_defaults(fn=get_weather)

agent2 = ReActAgent(llm=model, tools=[get_weather])
ctx2 = Context(agent2)
memory = ChatMemoryBuffer.from_defaults(token_limit=4000)

In [ ]:
resp = await agent2.run("长沙天气怎么样", ctx=ctx2, memory=memory)
print(resp.response)
resp = await agent2.run("合肥呢？", ctx=ctx2, memory=memory)
print(resp.response)

使用LLamaindex构建RAG知识库

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from langchain_openai.chat_models import ChatOpenAI
from Config.load_key import open_key
from Config.load_key import base_model
from Config.load_key import base_url
from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels, \
    DashScopeTextEmbeddingType

#初始化通义千问的embedding模型
Settings.embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V2,
    text_type=DashScopeTextEmbeddingType.TEXT_TYPE_DOCUMENT,
    api_key=open_key
)

Settings.llm = ChatOpenAI(
    model=base_model,
    base_url=base_url,
    api_key=open_key
)

#加载参考数据
document=SimpleDirectoryReader('Source').load_data()
index=VectorStoreIndex.from_documents(document)

#构建检索索引
query_engine=index.as_query_engine()
resp=query_engine.query('身高170，体重55kg，应该选什么衣服在南方冬天的时候，尽量穿的舒服一点')
resp
